In [ ]:
import pandas as pd

df = pd.read_csv("whisper_medium_predictions.csv")

df["split_clean"] = (
    df["split"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_true"] = (
    df["speech_present"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["speech_type_clean"] = (
    df["speech_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["pred_language"] = (
    df["whisper_med_prediction"]
    .astype("string")
    .str.strip()
    .str.lower()
)

def parse_languages(x):
    if pd.isna(x):
        return set()

    langs = {
        str(lang).strip().lower()
        for lang in str(x).split(";")
        if str(lang).strip()
    }

    langs -= {"neutral", "mixed", "unclear"}

    return {
        "chinese" if lang == "mandarin" else lang
        for lang in langs
    }

df["true_language_set"] = df["langs_present"].apply(parse_languages)
df["n_valid_languages"] = df["true_language_set"].map(len)

single = df[
    df["split_clean"].eq("dev")
    & df["speech_true"].eq("yes")
    & df["n_valid_languages"].eq(1)
    & df["speech_type_clean"].eq("live")
].copy()

single["true_language"] = single["true_language_set"].apply(
    lambda x: next(iter(x))
)

# cantonese excluded, as Whisper has no distinct Cantonese LID label

single = single[
    ~single["true_language"].eq("cantonese")
].copy()

single["correct"] = (
    single["pred_language"] == single["true_language"]
)

correct = single[single["correct"]]
incorrect = single[~single["correct"]]

print(f"N total: {len(single)}")
print(f"N correct: {len(correct)}")
print(f"N incorrect: {len(incorrect)}")
print()

print("CORRECT PREDICTIONS")
print(
    f"Mean confidence: "
    f"{correct['whisper_med_confidence'].mean():.3f}"
)
print(
    f"Median confidence: "
    f"{correct['whisper_med_confidence'].median():.3f}"
)
print()

print("INCORRECT PREDICTIONS")
print(
    f"Mean confidence: "
    f"{incorrect['whisper_med_confidence'].mean():.3f}"
)
print(
    f"Median confidence: "
    f"{incorrect['whisper_med_confidence'].median():.3f}"
)

print("\nCONFIDENCE THRESHOLDS")

for threshold in [0.50, 0.60, 0.70, 0.80, 0.90]:
    retained = single[
        single["whisper_med_confidence"] >= threshold
    ]

    if len(retained) == 0:
        continue

    accuracy = retained["correct"].mean() * 100
    coverage = len(retained) / len(single) * 100

    print(
        f"{threshold:.2f}: "
        f"accuracy={accuracy:.2f}% | "
        f"coverage={coverage:.2f}% | "
        f"n={len(retained)}"
    )


In [ ]:

import matplotlib.pyplot as plt

correct_conf = single.loc[
    single["correct"],
    "whisper_med_confidence"
].dropna()

incorrect_conf = single.loc[
    ~single["correct"],
    "whisper_med_confidence"
].dropna()

fig, ax = plt.subplots(figsize=(6, 5))

parts = ax.violinplot(
    [incorrect_conf, correct_conf],
    positions=[1, 2],
    showmeans=False,
    showmedians=False,
    showextrema=False
)

ax.boxplot(
    [incorrect_conf, correct_conf],
    positions=[1, 2],
    widths=0.18,
    showfliers=False
)

ax.set_xticks([1, 2])
ax.set_xticklabels(["Incorrect", "Correct"])

ax.set_ylabel("Confidence")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thresholds = np.arange(0.0, 0.96, 0.05)

accuracy = []
coverage = []

for threshold in thresholds:
    retained = single[
        single["whisper_med_confidence"] >= threshold
    ]

    if len(retained) == 0:
        accuracy.append(np.nan)
        coverage.append(0)
        continue

    accuracy.append(
        retained["correct"].mean() * 100
    )

    coverage.append(
        len(retained) / len(single) * 100
    )

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(
    coverage,
    accuracy,
    marker="o"
)

label_thresholds = {
    0.0, 0.4, 0.5, 0.6,
    0.7, 0.8, 0.9
}

for t, x, y in zip(thresholds, coverage, accuracy):
    if round(t, 2) in label_thresholds:
        ax.annotate(
            f"{t:.1f}",
            (x, y),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9
        )

ax.set_xlabel("Coverage (%)")
ax.set_ylabel("Accuracy (%)")

ax.set_xlim(20, 100)
ax.set_ylim(65, 100)

plt.tight_layout()
plt.show()



In [ ]:
import matplotlib.pyplot as plt

incorrect = single[
    ~single["correct"]
].copy()

wrong_predictions = (
    incorrect["pred_language"]
    .value_counts()
    .rename_axis("predicted_language")
    .reset_index(name="n_errors")
)

wrong_predictions["pct_of_all_errors"] = (
    wrong_predictions["n_errors"]
    / len(incorrect)
    * 100
)

print(
    wrong_predictions.head(15).to_string(
        index=False,
        formatters={
            "pct_of_all_errors": "{:.2f}".format
        }
    )
)

top_wrong = (
    wrong_predictions
    .head(10)
    .sort_values("pct_of_all_errors")
)

fig, ax = plt.subplots(figsize=(7, 5))

ax.barh(
    top_wrong["predicted_language"].str.title(),
    top_wrong["pct_of_all_errors"]
)

ax.set_xlabel("Percentage of incorrect predictions (%)")
ax.set_ylabel("Predicted language")

plt.tight_layout()
plt.show()
